# Tutorial 1_1: 1D bar

In this tutorial, we will model a simplified 1D problem: A structural member passing through a hot region. The bar experiences a uniform temperature increase, but its expansion is prevented because it is rigidly fixed at both ends to the main structure.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt

# Mesh definition

To describe the deformation of a body, we use the displacement vector field $u(x)$. For this idealized 1D model, we define a bar of length $L$. We assume small displacements.

In [ ]:
Lx = 0.5
nx = 50

mesh = IntervalMesh(nx, Lx)

V = FunctionSpace(mesh, 'CG', 1)
DG0 = FunctionSpace(mesh, 'DG', 0)

x_nodes = mesh.coordinates.dat.data
x_elements = (x_nodes[:-1] + x_nodes[1:]) / 2.0

# Material Properties and Problem Definition

Thermoelastic responses depend strongly on material properties, specifically Young's modulus ($E$) and the coefficient of thermal expansion ($\alpha$). For this problem, we apply a uniform temperature change ($\Delta T$) along the bar. The free thermal strain in this scenario, if unconstrained, would be $\alpha \Delta T$. Because the bar is rigidly fixed at both ends, we apply Dirichlet (displacement) boundary conditions where $u = 0$ at the boundaries $\Gamma_u$.

In [ ]:
alpha = Constant(1.2e-5)
E = Constant(210e9)
delta_T = Constant(200)

u = Function(V, name='Displacement [m]')
T = Function(V, name='Temperature [C]')
stress = Function(DG0, name='Stress [MPa]')

T.assign(delta_T)

bc_mech = [DirichletBC(V, Constant(0), 1),
           DirichletBC(V, Constant(0), 2)]

# Solver Setup
We aim to solve the quasi-static equilibrium equation $\sigma_{ij,j} + b_i = 0$ in the absence of body forces.

## Constitutive Law (1D):
In linear thermoelasticity, stress depends on how much the total strain differs from the free thermal strain. For our 1D bar, the stress is defined as:
$$\sigma = E(\varepsilon(u) - \alpha \Delta T)$$Where $\varepsilon(u)$ is the small-strain measure.

## Variational (Weak) Form:

Using the Principle of Virtual Work, we multiply by a test function $v$ and integrate by parts to obtain the weak form:$$\int_{\Omega} \sigma(u, T) : \varepsilon(v) \, dx = 0$$

In [ ]:
def eps(u):
    return u.dx(0)

def sigma_1d(u, T):
    return E * (eps(u) - alpha * T)

u_mech = TrialFunction(V)
v_mech = TestFunction(V)

mech_form = sigma_1d(u_mech, T) * eps(v_mech) * dx

a_mech = lhs(mech_form)
L_mech = rhs(mech_form)

mech_prob = LinearVariationalProblem(a_mech, L_mech, u, bcs=bc_mech, constant_jacobian=True)
mech_solver = LinearVariationalSolver(mech_prob)

# Solution and analytical validation

Because the expansion is fully constrained, the free thermal strain is converted entirely into significant thermal stress. Let's solve the system and compare our finite element result to the analytical expectation for a fully constrained 1D bar ($\sigma = E \alpha \Delta T$).

In [ ]:
mech_solver.solve()

stress.project(abs(sigma_1d(u, T)) / 1e6)

computed_stress = stress.dat.data[0]
expected_sigma = float(E) * float(alpha) * float(delta_T) / 1e6

print(f"Theoretical 1D Stress: {expected_sigma:.1f} MPa")
print(f"Firedrake Computed Stress: {computed_stress:.1f} MPa")

plt.figure(figsize=(8, 2))
plt.plot(x_elements, stress.dat.data, label="Thermal Stress")
plt.xlabel("Position x [m]")
plt.ylabel("Stress [MPa]")
plt.title("1D Axial Thermal Stress")
plt.legend()
plt.show()

# Free Expansion with a Spatially Varying Temperature Field

In the previous example, the bar was fully constrained, converting all thermal strain into thermal stress. What happens if we remove the constraint on one end and apply a spatially varying temperature field?

According to linear thermoelasticity, if a statically determinate structure (like a bar fixed at only one end) is heated and free to expand, no mechanical stresses are generated. The material will simply experience a free thermal strain.

Let's model this by:

Releasing the right boundary (leaving it traction-free).

Applying a linear temperature gradient $T(x)$ that is 0°C at the fixed end ($x=0$) and reaches $\Delta T$ at the free end ($x=L$).

In [ ]:
# Create a spatial coordinate to define x-dependent functions
x_coord = SpatialCoordinate(mesh)[0]

# Define a linear temperature field: T(x) = delta_T * (x / Lx)
T_linear_expr = delta_T * (x_coord / Lx)
T.interpolate(T_linear_expr)

# Solving the free expansion problem

We use the exact same constitutive law and weak form as before. We only swap in our new spatially varying temperature field and our updated boundary condition.

Because the right end is free, the bar will expand to accommodate the free thermal strain:$$\varepsilon_{\theta} = \alpha \Delta T(x)$$The analytical displacement $u(x)$ for a linear temperature field $T(x) = \Delta T \frac{x}{L}$ is obtained by integrating the thermal strain:$$u(x) = \int_0^x \alpha \Delta T \frac{\xi}{L} \, d\xi = \frac{\alpha \Delta T}{2L} x^2$$

What stress values can we expect?

In [ ]:
u_mech_free = TrialFunction(V)
v_mech_free = TestFunction(V)

mech_form_free = sigma_1d(u_mech_free, T) * eps(v_mech_free) * dx

a_free = lhs(mech_form_free)
L_free = rhs(mech_form_free)

free_prob = LinearVariationalProblem(a_free, L_free, u, bcs=bc_mech[0], constant_jacobian=True)
free_solver = LinearVariationalSolver(free_prob)
free_solver.solve()

stress.project(sigma_1d(u, T) / 1e6)

analytical_u_max = 0.5 * float(alpha) * float(delta_T) * Lx
computed_u_max = u.dat.data[-1]

print(f"Analytical Max Displacement: {analytical_u_max:.5e} m")
print(f"Firedrake Max Displacement:  {computed_u_max:.5e} m")
print(f"Maximum Computed Stress:     {max(abs(stress.dat.data)):.5e} MPa")

# Plotting the results
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Plot Temperature
axs[0].plot(x_nodes, T.dat.data, color='red')
axs[0].set_title("Temperature Field T(x)")
axs[0].set_xlabel("Position x [m]")
axs[0].set_ylabel("Temperature [C]")

# Plot Displacement
axs[1].plot(x_nodes, u.dat.data, color='blue', label="Firedrake")
x_exact = np.linspace(0, Lx, 100)
u_exact = (float(alpha) * float(delta_T) / (2 * Lx)) * x_exact**2
axs[1].plot(x_exact, u_exact, 'k--', label="Analytical")
axs[1].set_title("Displacement u(x)")
axs[1].set_xlabel("Position x [m]")
axs[1].set_ylabel("Displacement [m]")
axs[1].legend()

# Plot Stress
axs[2].plot(x_elements, stress.dat.data, color='green')
axs[2].set_title("Thermal Stress")
axs[2].set_xlabel("Position x [m]")
axs[2].set_ylabel("Stress [MPa]")
axs[2].set_ylim(-1, 1)

plt.tight_layout()
plt.show()

If the bar is fixed at only one end, the thermal stress remains exactly zero everywhere, even if the temperature field is highly non-linear. There are still no mechanical constraints resisting the local expansion of the material, so the bar is completely free to grow or shrink point-by-point according to the local temperature.

In 1D, this means thermal stresses will always be zero if the body is unconstrained, simply because in this case compatibility is always ensured. In higher dimensions, a non-uniform temperature field can cause thermal stresses even in a free body because different parts of the material try to expand in ways that don't fit together geometrically, and elastic stresses appear to enforce the compatibility condition.